# Food Vision: fine-tuning (125 classes, Food-101 + TurkishFoods-25)

Improves the shipped model (`best_resnet50_foodvision.pth`: frozen backbone + `layer4`, 77.1 % test top-1) by
fine-tuning **every layer** with stronger augmentation and label smoothing, on the **same splits** as the first
notebook, so the results are directly comparable.

**How to run**
1. *Runtime → Change runtime type → T4 GPU.*
2. Upload the current `best_resnet50_foodvision.pth` to `MyDrive/foodvision/` (warm start; optional but faster).
3. *Runtime → Run all.* Allow Google Drive when asked.
4. Checkpoints go to Google Drive after every epoch. If Colab disconnects, open the notebook again and *Run all*:
   training continues from the last finished epoch.
5. At the end the test results (top-1, top-3, Food-101 / Turkish, macro-F1, hardest classes) are printed and saved
   next to the weights. Bring `best_<arch>_foodvision.pth` and `results_<arch>.json` back to compare.

To try another architecture, change `ARCH` in the next cell and run again (each run has its own Drive folder).
Note: the service (`app.py`) loads a **ResNet50**; another winner would need a one-line change there.

In [ ]:
# ---- Settings -------------------------------------------------------------------------------------------------
ARCH = "resnet50"            # "resnet50" | "efficientnet_v2_s" | "convnext_tiny"
EPOCHS = 8                   # full fine-tuning epochs (early stopping may end sooner)
PATIENCE = 3                 # stop after this many epochs without a better validation top-1
BATCH_SIZE = 64
BACKBONE_LR = 1e-4           # pretrained layers
HEAD_LR = 1e-3               # the 125-class layer
WEIGHT_DECAY = 0.05
LABEL_SMOOTHING = 0.1
NUM_WORKERS = 0              # 2 is faster but Colab can run out of memory ("DataLoader worker exited unexpectedly")
SEED = 42

DRIVE_ROOT = "/content/drive/MyDrive/foodvision"
WARM_START = f"{DRIVE_ROOT}/best_resnet50_foodvision.pth"   # used only for ARCH == "resnet50", if the file exists
RUN_DIR = f"{DRIVE_ROOT}/run_{ARCH}"

In [ ]:
!pip install -q datasets

import json, math, os, random, time
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
from datasets import load_dataset
from google.colab import drive
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm

drive.mount("/content/drive")
os.makedirs(RUN_DIR, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert device.type == "cuda", "No GPU: Runtime -> Change runtime type -> T4 GPU"
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)

## Data: same splits as the first notebook
- Food-101 official train (75 750) → 90 % train / 10 % validation, stratified, `random_state=42`.
- Food-101 official test (Hugging Face `validation`, 25 250) → test.
- TurkishFoods-25 keeps its own train / eval / test.
- 125 classes = sorted union; *baklava* is in both and becomes one class.
- Labels are matched **by name** (the Hugging Face label order is not the sorted order).

In [ ]:
food101 = load_dataset("ethz/food101")
turkish = load_dataset("yunusserhat/TurkishFoods-25")

food101_names = food101["train"].features["label"].names
turkish_names = turkish["train"].features["label"].names
classes = sorted(set(food101_names) | set(turkish_names))
assert len(classes) == 125
class_to_idx = {name: i for i, name in enumerate(classes)}
TURKISH = set(turkish_names) - set(food101_names)          # 24 Turkish-only classes (baklava counts as Food-101)

with open(f"{RUN_DIR}/food_classes.json", "w", encoding="utf-8") as f:
    json.dump(classes, f, ensure_ascii=False, indent=2)

def rows(data, split, names, source):
    return [(source, split, i, class_to_idx[names[label]]) for i, label in enumerate(data[split]["label"])]

f101_train_all = rows(food101, "train", food101_names, "food101")
f101_train, f101_val = train_test_split(
    f101_train_all, test_size=0.10, random_state=42, stratify=[r[3] for r in f101_train_all])

train_rows = f101_train + rows(turkish, "train", turkish_names, "turkish")
val_rows = f101_val + rows(turkish, "eval", turkish_names, "turkish")
test_rows = rows(food101, "validation", food101_names, "food101") + rows(turkish, "test", turkish_names, "turkish")
print("train", len(train_rows), "| val", len(val_rows), "| test", len(test_rows))

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.5, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.TrivialAugmentWide(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=0.25),
])
# Exactly what food-vision-service/app.py does at prediction time.
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

SOURCES = {"food101": food101, "turkish": turkish}

class FoodRows(Dataset):
    def __init__(self, rows, transform):
        self.rows, self.transform = rows, transform
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        source, split, index, label = self.rows[i]
        image = SOURCES[source][split][index]["image"].convert("RGB")
        return self.transform(image), label

def loader(rows, transform, shuffle):
    return DataLoader(FoodRows(rows, transform), batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=NUM_WORKERS,
                      pin_memory=True, persistent_workers=NUM_WORKERS > 0, drop_last=shuffle)

train_loader = loader(train_rows, train_transform, True)
val_loader = loader(val_rows, eval_transform, False)
test_loader = loader(test_rows, eval_transform, False)

# Same class weights as the first notebook: sqrt(mean / count), normalised to a mean of 1.
counts = Counter(r[3] for r in train_rows)
count_tensor = torch.tensor([counts[i] for i in range(125)], dtype=torch.float32)
class_weights = torch.sqrt(count_tensor.mean() / count_tensor)
class_weights = (class_weights / class_weights.mean()).to(device)

## Model

In [ ]:
def build(arch):
    if arch == "resnet50":
        model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        model.fc = nn.Linear(model.fc.in_features, 125)
        if os.path.exists(WARM_START):
            model.load_state_dict(torch.load(WARM_START, map_location="cpu", weights_only=True))
            print("Warm start from", WARM_START)
        head = model.fc
    elif arch == "efficientnet_v2_s":
        model = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.DEFAULT)
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, 125)
        head = model.classifier
    elif arch == "convnext_tiny":
        model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, 125)
        head = model.classifier
    else:
        raise ValueError(arch)
    return model, head

model, head = build(ARCH)
model = model.to(device).to(memory_format=torch.channels_last)

head_ids = {id(p) for p in head.parameters()}
optimizer = torch.optim.AdamW([
    {"params": [p for p in model.parameters() if id(p) not in head_ids], "lr": BACKBONE_LR},
    {"params": list(head.parameters()), "lr": HEAD_LR},
], weight_decay=WEIGHT_DECAY)

# One epoch of linear warm-up, then cosine decay to 1 % of the starting rate, per step.
steps_per_epoch = len(train_loader)
total_steps, warmup_steps = EPOCHS * steps_per_epoch, steps_per_epoch
def lr_factor(step):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * progress))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
scaler = torch.amp.GradScaler("cuda")

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"{ARCH}: {trainable:,} trainable parameters, {steps_per_epoch} steps per epoch")

## Train (resumes from Google Drive)

In [ ]:
LAST = f"{RUN_DIR}/last_checkpoint.pth"
BEST = f"{RUN_DIR}/best_{ARCH}_foodvision.pth"

start_epoch, best_val, history, bad_epochs = 0, 0.0, [], 0
if os.path.exists(LAST):
    state = torch.load(LAST, map_location="cpu", weights_only=False)
    model.load_state_dict(state["model"]); optimizer.load_state_dict(state["optimizer"])
    scheduler.load_state_dict(state["scheduler"]); scaler.load_state_dict(state["scaler"])
    start_epoch, best_val, history, bad_epochs = (state["epoch"], state["best_val"], state["history"],
                                                  state["bad_epochs"])
    print(f"Resuming after epoch {start_epoch} (best validation top-1 {best_val:.2f} %)")

@torch.inference_mode()
def accuracy(loader):
    model.eval()
    top1 = top3 = total = 0
    for images, labels in tqdm(loader, leave=False):
        images = images.to(device, non_blocking=True).to(memory_format=torch.channels_last)
        labels = labels.to(device, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16):
            best = model(images).topk(3, dim=1).indices
        top1 += (best[:, 0] == labels).sum().item()
        top3 += (best == labels[:, None]).any(dim=1).sum().item()
        total += labels.size(0)
    return 100 * top1 / total, 100 * top3 / total

# No comparison with the shipped model on validation: this validation split is drawn from the Hugging Face Food-101
# train list, whose order differs from the Kaggle list the shipped model was trained on, so most of these photos were
# in its training data (it scores ~92 % here instead of 72.5 %). The fair comparison is the test set, at the end.

for epoch in range(start_epoch, EPOCHS):
    if bad_epochs >= PATIENCE:
        print("Early stopping: no better validation top-1 for", PATIENCE, "epochs")
        break
    model.train()
    seen = correct = loss_sum = 0
    started = time.time()
    for images, labels in tqdm(train_loader, desc=f"epoch {epoch + 1}/{EPOCHS}"):
        images = images.to(device, non_blocking=True).to(memory_format=torch.channels_last)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.float16):
            outputs = model(images)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update(); scheduler.step()
        seen += labels.size(0); correct += (outputs.argmax(1) == labels).sum().item()
        loss_sum += loss.item() * labels.size(0)

    val1, val3 = accuracy(val_loader)
    improved = val1 > best_val
    if improved:
        best_val, bad_epochs = val1, 0
        torch.save(model.state_dict(), BEST)
    else:
        bad_epochs += 1
    history.append({"epoch": epoch + 1, "train_loss": loss_sum / seen, "train_top1": 100 * correct / seen,
                    "val_top1": val1, "val_top3": val3, "minutes": (time.time() - started) / 60})
    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(), "epoch": epoch + 1,
                "best_val": best_val, "history": history, "bad_epochs": bad_epochs}, LAST)
    print(f"epoch {epoch + 1}: train loss {loss_sum / seen:.3f} | val top-1 {val1:.2f} % top-3 {val3:.2f} % | "
          f"{'NEW BEST, saved' if improved else 'no improvement'} | {(time.time() - started) / 60:.1f} min")

print("Best validation top-1:", round(best_val, 2), "%")

## Test: same test set and metrics as the shipped model

In [ ]:
if not os.path.exists(BEST):
    raise SystemExit(f"{BEST} is missing: the training cell did not finish an epoch.")
model.load_state_dict(torch.load(BEST, map_location=device, weights_only=True))
model.to(device).eval()   # the speed check below moves the model to the CPU; running this cell again must undo that

truth, top1, top3 = [], [], []
with torch.inference_mode():
    for images, labels in tqdm(test_loader, desc="test"):
        images = images.to(device).to(memory_format=torch.channels_last)
        with torch.autocast("cuda", dtype=torch.float16):
            best = model(images).topk(3, dim=1).indices.cpu()
        truth += labels.tolist(); top1 += best[:, 0].tolist(); top3 += [l in row for l, row in zip(labels.tolist(), best.tolist())]

turkish_ids = {class_to_idx[n] for n in TURKISH}
def part(keep):
    idx = [i for i, t in enumerate(truth) if keep(t)]
    labels = {truth[i] for i in idx}
    f1 = []
    for c in labels:
        tp = sum(1 for i in idx if truth[i] == c and top1[i] == c)
        fp = sum(1 for i in idx if truth[i] != c and top1[i] == c)
        fn = sum(1 for i in idx if truth[i] == c and top1[i] != c)
        f1.append(0 if tp == 0 else 2 * tp / (2 * tp + fp + fn))
    return {"images": len(idx), "top1": round(100 * sum(top1[i] == truth[i] for i in idx) / len(idx), 2),
            "top3": round(100 * sum(top3[i] for i in idx) / len(idx), 2), "macro_f1": round(100 * np.mean(f1), 2)}

per_class = []
for c in range(125):
    idx = [i for i, t in enumerate(truth) if t == c]
    per_class.append((round(100 * sum(top1[i] == c for i in idx) / len(idx), 1), classes[c]))
per_class.sort()

model.to("cpu"); photo = torch.zeros(1, 3, 224, 224)
with torch.inference_mode():
    model(photo); t = time.perf_counter()
    for _ in range(10): model(photo)
cpu_ms = round(100 * (time.perf_counter() - t), 1)

results = {"model": ARCH, "num_classes": 125, "validation_accuracy": round(best_val, 2),
           "all": part(lambda t: True), "food101": part(lambda t: t not in turkish_ids),
           "turkish": part(lambda t: t in turkish_ids), "hardest_classes": per_class[:10],
           "cpu_ms_per_photo": cpu_ms, "size_mb": round(os.path.getsize(BEST) / 2**20, 1), "history": history}
with open(f"{RUN_DIR}/results_{ARCH}.json", "w") as f:
    json.dump(results, f, indent=2)

print(json.dumps({k: v for k, v in results.items() if k != "history"}, indent=2))
print("\nShipped model for comparison: top-1 77.09 % | Food-101 76.73 % | Turkish 85.09 %")

In [ ]:
from google.colab import files
files.download(BEST)
files.download(f"{RUN_DIR}/results_{ARCH}.json")